In [ ]:
import sys
from pathlib import Path

# Notebook lives two levels below project root (notebooks/microglia.ipynb/)
project_root = Path().resolve().parent.parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('Libraries loaded successfully!')
out_dir = Path('../../outputs/GSE103334')
out_dir.mkdir(parents=True, exist_ok=True)
%load_ext autoreload


In [ ]:
# Install GEOparse if not already installed
# !pip install GEOparse

import GEOparse

# Download GSE103334
print("Downloading data from GEO (this may take a few minutes)...")
gse = GEOparse.get_GEO(geo="GSE103334", destdir="../../data/")

print(f"\nGSE Title: {gse.metadata['title'][0]}")
print(f"Number of samples: {len(gse.gsms)}")
print(f"\nSummary: {gse.metadata['summary'][0][:500]}...")

In [ ]:
# Extract sample metadata
sample_info = []
for gsm_name, gsm in gse.gsms.items():
    sample_info.append({
        'gsm_id': gsm_name,
        'title': gsm.metadata.get('title', [''])[0],
        'source': gsm.metadata.get('source_name_ch1', [''])[0],
        'treatment': gsm.metadata.get('treatment_protocol_ch1', [''])[0],
    })

metadata_df = pd.DataFrame(sample_info)
print(f"Total samples: {len(metadata_df)}")
metadata_df.head(10)

In [ ]:
# List available supplementary files
print("Supplementary files:")
for supp_file in gse.metadata.get('supplementary_file', []):
    print(f"  - {supp_file}")

In [ ]:
# Load expression data
# Look for expression matrix files (common formats: .txt, .csv, .tsv, .txt.gz)

data_dir = Path("../../data/GSE103334")
data_dir.mkdir(parents=True, exist_ok=True)

expression_files = list(data_dir.glob("*.txt*")) + list(data_dir.glob("*.csv*")) + list(data_dir.glob("*.tsv*"))

print(f"Found {len(expression_files)} potential expression files:")
for f in expression_files[:10]:
    print(f"  - {f.name}")

# Try to load the first one to see the format
if expression_files:
    test_file = expression_files[0]
    print(f"\nAttempting to load: {test_file.name}")
    
    try:
        if test_file.suffix == '.gz':
            df = pd.read_csv(test_file, compression='gzip', sep='\t', nrows=5)
        else:
            df = pd.read_csv(test_file, sep='\t', nrows=5)
        
        print(f"\nPreview of {test_file.name}:")
        print(f"Shape (first 5 rows): {df.shape}")
        print(df)
    except Exception as e:
        print(f"Error loading file: {e}")
        print("Trying with different separator...")
        try:
            if test_file.suffix == '.gz':
                df = pd.read_csv(test_file, compression='gzip', nrows=5)
            else:
                df = pd.read_csv(test_file, nrows=5)
            print(df)
        except Exception as e2:
            print(f"Still failed: {e2}")
else:
    print("\nNo expression files found. The data might be in a different format.")
    print("Let's check the GEO website directly...")

In [ ]:
# Load the expression matrix
data_file = Path("../../data/GSE103334/GSE103334_FPKM_CKP25_TOPHAT.txt.gz")

print("Loading expression data...")
expr_df = pd.read_csv(data_file, sep='\t', compression='gzip', index_col=0)

print(f"Expression matrix shape: {expr_df.shape}")
print(f"  - {expr_df.shape[0]:,} genes")
print(f"  - {expr_df.shape[1]:,} cells")
print(f"\nFirst few genes and cells:")
expr_df.iloc[:5, :5]

In [ ]:
# Parse cell metadata from column names
# Format: CK_0w_m3_A1 -> CK (condition), 0w (time point), m3 (mouse), A1 (well)

cell_metadata = []
for cell_name in expr_df.columns:
    parts = cell_name.split('_')
    if len(parts) >= 4:
        cell_metadata.append({
            'cell_id': cell_name,
            'condition': parts[0],
            'time_point': parts[1],
            'mouse': parts[2],
            'well': parts[3]
        })

cell_meta_df = pd.DataFrame(cell_metadata)

print(f"Parsed metadata for {len(cell_meta_df)} cells")
print(f"\nTime points: {sorted(cell_meta_df['time_point'].unique())}")
print(f"Conditions: {cell_meta_df['condition'].unique()}")
print(f"Number of mice: {cell_meta_df['mouse'].nunique()}")

# Count cells per time point
print(f"\nCells per time point:")
print(cell_meta_df['time_point'].value_counts().sort_index())

In [ ]:
# Calculate QC metrics
cell_meta_df['total_counts'] = expr_df.sum(axis=0).values
cell_meta_df['n_genes_detected'] = (expr_df > 0).sum(axis=0).values

# Plot QC metrics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total expression per cell
axes[0].hist(cell_meta_df['total_counts'], bins=50, edgecolor='black')
axes[0].set_xlabel('Total FPKM per cell')
axes[0].set_ylabel('Number of cells')
axes[0].set_title('Distribution of Total Expression')
axes[0].axvline(cell_meta_df['total_counts'].median(), color='red', linestyle='--', label='Median')
axes[0].legend()

# Number of genes detected per cell
axes[1].hist(cell_meta_df['n_genes_detected'], bins=50, edgecolor='black')
axes[1].set_xlabel('Number of genes detected (FPKM > 0)')
axes[1].set_ylabel('Number of cells')
axes[1].set_title('Genes Detected per Cell')
axes[1].axvline(cell_meta_df['n_genes_detected'].median(), color='red', linestyle='--', label='Median')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Median total FPKM per cell: {cell_meta_df['total_counts'].median():.2f}")
print(f"Median genes detected per cell: {cell_meta_df['n_genes_detected'].median():.0f}")

In [ ]:
# Check how many cells per mouse and time point
print("Cells per mouse:")
print(cell_meta_df.groupby(['time_point', 'mouse']).size().unstack(fill_value=0))

print("\n\nTotal cells per mouse (across all time points):")
print(cell_meta_df['mouse'].value_counts().sort_index())

# Well-plate format detection — use the already-parsed 'well' column
# (some cell names have 5 parts, e.g. CKp25_6w_m4_H9_0, so split('_')[-1] is unreliable)
print("\n\nWell positions used (to see if it's 96-well or 384-well):")
wells = cell_meta_df['well'].values
well_letters = [w[0]  for w in wells if len(w) >= 2 and w[0].isalpha()]
well_numbers = [int(w[1:]) for w in wells if len(w) >= 2 and w[0].isalpha() and w[1:].isdigit()]

print(f"Row letters : {sorted(set(well_letters))}")
print(f"Column range: {min(well_numbers)} – {max(well_numbers)}")

if well_letters and well_numbers:
    if max(well_numbers) <= 12 and max(well_letters) <= 'H':
        print("\n→ 96-well plate format (8 rows × 12 columns)")
    elif max(well_numbers) <= 24 and max(well_letters) <= 'P':
        print("\n→ 384-well plate format (16 rows × 24 columns)")
    else:
        print(f"\n→ Custom plate format")


In [ ]:
# Define key microglia marker genes
microglia_markers = {
    'Homeostatic': ['Cx3cr1', 'Tmem119', 'P2ry12', 'Gpr34', 'Fcrls', 'Siglech'],
    'Pan-microglia': ['Aif1', 'Csf1r', 'Itgam', 'Hexb'],  # Aif1 = Iba1, Itgam = Cd11b
    'Activated/DAM': ['Apoe', 'Trem2', 'Tyrobp', 'Axl', 'Cd68', 'Lpl', 'Cst7'],  # DAM = Disease-Associated Microglia
    'Inflammatory': ['Il1b', 'Tnf', 'Il6', 'Nos2', 'Ccl2', 'Ccl3', 'Ccl4'],
    'Complement': ['C1qa', 'C1qb', 'C1qc'],
    'Phagocytosis': ['Cd68', 'Cd74', 'Clec7a', 'Mrc1'],
}

# Check which markers are present in the data
print("Marker genes present in dataset:\n")
all_genes = set(expr_df.index)
for category, genes in microglia_markers.items():
    present = [g for g in genes if g in all_genes]
    missing = [g for g in genes if g not in all_genes]
    print(f"{category}:")
    print(f"  ✓ Present ({len(present)}): {', '.join(present)}")
    if missing:
        print(f"  ✗ Missing ({len(missing)}): {', '.join(missing)}")
    print()

In [ ]:
# Find most variable genes (likely most biologically relevant)
gene_var = expr_df.var(axis=1)
gene_mean = expr_df.mean(axis=1)

# Filter out very low expression genes
expressed_genes = gene_mean[gene_mean > 1].index
gene_var_filtered = gene_var[expressed_genes]

# Top 20 most variable genes
top_variable = gene_var_filtered.nlargest(20)

print("Top 20 most variable genes (excluding low expression):\n")
for i, (gene, variance) in enumerate(top_variable.items(), 1):
    mean_expr = gene_mean[gene]
    print(f"{i:2d}. {gene:15s} - var: {variance:8.2f}, mean: {mean_expr:6.2f}")

# Check if these overlap with our markers
marker_genes_flat = [g for genes in microglia_markers.values() for g in genes]
overlap = set(top_variable.index) & set(marker_genes_flat)
print(f"\n{len(overlap)} of top 20 variable genes are known microglia markers: {overlap}")

## Counterfactual Reconstruction and Differential Expression Analysis

Reconstruct what CKp25 microglia would look like under CK (control) dynamics using
Wasserstein parallel transport, then identify genes whose expression differs significantly
between the observed CKp25 trajectory and the counterfactual (CF).

In [ ]:
import scanpy as sc

# Build AnnData from already-loaded expr_df
adata_time = sc.AnnData(X=expr_df.T.values)
adata_time.obs_names = expr_df.columns.tolist()
adata_time.var_names = expr_df.index.tolist()
adata_time.obs['condition']  = cell_meta_df.set_index('cell_id').loc[expr_df.columns, 'condition'].values
adata_time.obs['time_point'] = cell_meta_df.set_index('cell_id').loc[expr_df.columns, 'time_point'].values
adata_time.obs['mouse']      = cell_meta_df.set_index('cell_id').loc[expr_df.columns, 'mouse'].values

# Preprocessing: log2 → gene filter → HVG → PCA → kNN graph → UMAP
adata_pp = adata_time.copy()
adata_pp.X = np.log2(adata_pp.X + 1)

sc.pp.filter_genes(adata_pp, min_cells=10)
print(f'Genes after low-expression filter : {adata_pp.n_vars:,}')

sc.pp.highly_variable_genes(adata_pp, n_top_genes=2000, flavor='seurat')
print(f'Highly variable genes             : {adata_pp.var["highly_variable"].sum()}')

sc.tl.pca(adata_pp, use_highly_variable=True, n_comps=50, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata_pp, log=True, n_pcs=50)

sc.pp.neighbors(adata_pp, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata_pp, random_state=42)
print('UMAP computed.')


In [ ]:
# UMAP coloured by time point, condition, and mouse
time_order  = ['0w', '1w', '2w', '6w']
present_tps = [tp for tp in time_order if tp in adata_pp.obs['time_point'].values]
adata_pp.obs['time_point'] = pd.Categorical(
    adata_pp.obs['time_point'], categories=present_tps, ordered=True
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sc.pl.umap(adata_pp, color='time_point', ax=axes[0], show=False,
           title='Time point', palette='plasma', frameon=False)
sc.pl.umap(adata_pp, color='condition',  ax=axes[1], show=False,
           title='Condition (CK vs CKp25)',
           palette=['#2E86AB', '#D62828'], frameon=False)
sc.pl.umap(adata_pp, color='mouse',      ax=axes[2], show=False,
           title='Mouse', frameon=False)

plt.suptitle('UMAP of microglia scRNA-seq (Mathys et al. 2017)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(out_dir / 'umap_time_condition_mouse.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
%autoreload 2
from src.cf_recon import reconstruct_cf

# Standardise top n_pcs to unit variance (keeps PT numerics O(1))
n_pcs   = 10
pca_raw = adata_pp.obsm['X_pca'][:, :n_pcs]
pca_std = pca_raw.std(axis=0)
pca     = pca_raw / pca_std
obs     = adata_pp.obs

def get_pca(cond, tp):
    mask = (obs['condition'] == cond) & (obs['time_point'] == tp)
    return pca[mask.values]

all_tps    = ['0w', '1w', '2w', '6w']
common_tps = [tp for tp in all_tps
              if len(get_pca('CK', tp)) > 0 and len(get_pca('CKp25', tp)) > 0]
n_steps    = len(common_tps) - 1

control_samples = [get_pca('CK',    tp) for tp in common_tps]
ckp25_samples   = [get_pca('CKp25', tp) for tp in common_tps]
cf_0            = get_pca('CKp25', common_tps[0])

print(f'Time points : {common_tps}  (n_steps={n_steps})')
for tp, ck, cp in zip(common_tps, control_samples, ckp25_samples):
    print(f'  {tp}: CK={len(ck)}, CKp25={len(cp)}')

print(f'\nRunning reconstruct_cf ({n_steps} steps, {n_pcs} PCs) ...')
cf_curve = reconstruct_cf(control_samples, cf_0, n=n_steps, project=True, tol=1e-6)
print('Done.')

# ── Visualise trajectories in PC1-PC2 ────────────────────────────────────────
COL_CK   = '#2E86AB'
COL_CP25 = '#D62828'
COL_CF   = '#2DC653'

ck_means    = np.array([s.mean(0)      for s in control_samples])
ckp25_means = np.array([s.mean(0)      for s in ckp25_samples])
cf_means    = np.array([m.locs.mean(0) for m in cf_curve])

fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharex=True, sharey=True)
for ax in axes:
    ax.set_xlabel('PC 1 (standardized)', fontsize=12)
    ax.set_ylabel('PC 2 (standardized)', fontsize=12)

# Left: CK vs CKp25
for i in range(len(common_tps)):
    axes[0].scatter(control_samples[i][:, 0], control_samples[i][:, 1],
                    c=COL_CK,   alpha=0.2, s=7, linewidths=0, rasterized=True)
    axes[0].scatter(ckp25_samples[i][:, 0],   ckp25_samples[i][:, 1],
                    c=COL_CP25, alpha=0.2, s=7, linewidths=0, rasterized=True)
axes[0].plot(ck_means[:, 0],    ck_means[:, 1],    'o-', c=COL_CK,   lw=2, ms=9, label='CK')
axes[0].plot(ckp25_means[:, 0], ckp25_means[:, 1], 's-', c=COL_CP25, lw=2, ms=9, label='CKp25')
for i, tp in enumerate(common_tps):
    axes[0].annotate(tp, ck_means[i, :2],    xytext=(-10, 5), textcoords='offset points', fontsize=9, color=COL_CK)
    axes[0].annotate(tp, ckp25_means[i, :2], xytext=(  5, 5), textcoords='offset points', fontsize=9, color=COL_CP25)
axes[0].set_title('Actual trajectories: CK vs CKp25', fontsize=13)
axes[0].legend(fontsize=11)

# Right: faded CK/CKp25 + CF
for i in range(len(common_tps)):
    axes[1].scatter(control_samples[i][:, 0], control_samples[i][:, 1],
                    c=COL_CK,   alpha=0.08, s=7, linewidths=0, rasterized=True)
    axes[1].scatter(ckp25_samples[i][:, 0],   ckp25_samples[i][:, 1],
                    c=COL_CP25, alpha=0.08, s=7, linewidths=0, rasterized=True)
    axes[1].scatter(cf_curve[i].locs[:, 0],   cf_curve[i].locs[:, 1],
                    c=COL_CF,   alpha=0.25, s=7, linewidths=0, rasterized=True)
axes[1].plot(ck_means[:, 0],    ck_means[:, 1],    'o-', c=COL_CK,   lw=2, ms=9, alpha=0.4, label='CK')
axes[1].plot(ckp25_means[:, 0], ckp25_means[:, 1], 's-', c=COL_CP25, lw=2, ms=9, alpha=0.4, label='CKp25 (actual)')
axes[1].plot(cf_means[:, 0],    cf_means[:, 1],    '^-', c=COL_CF,   lw=2, ms=9,            label='CF (CKp25 under CK dynamics)')
for i, tp in enumerate(common_tps):
    axes[1].annotate(tp, cf_means[i, :2], xytext=(5, 5), textcoords='offset points', fontsize=9, color=COL_CF)
axes[1].set_title('Reconstructed CF vs actual trajectories', fontsize=13)
axes[1].legend(fontsize=11)

plt.suptitle('Counterfactual reconstruction: CKp25 under CK dynamics\n'
             '(Mathys et al. 2017, Wasserstein parallel transport)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(out_dir / 'cf_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
from umap import UMAP

# Stack CK, CKp25, and CF PCA coordinates across all time points
all_locs, traj_label, tp_label = [], [], []
for i, tp in enumerate(common_tps):
    for locs, name in [
        (control_samples[i], 'CK'),
        (ckp25_samples[i],   'CKp25'),
        (cf_curve[i].locs,   'CF'),
    ]:
        all_locs.append(locs)
        traj_label.extend([name] * len(locs))
        tp_label.extend([tp]   * len(locs))

all_locs   = np.vstack(all_locs)
traj_label = np.array(traj_label)
tp_label   = np.array(tp_label)
print(f'Total points for joint UMAP: {len(all_locs):,}')

reducer   = UMAP(n_components=2, n_neighbors=30, min_dist=0.3, random_state=42)
embedding = reducer.fit_transform(all_locs)
u1, u2    = embedding[:, 0], embedding[:, 1]
print('UMAP done.')

# ── Plot: one panel per trajectory, coloured by time point ───────────────────
tp_colors = dict(zip(common_tps,
                     [plt.cm.plasma(v) for v in np.linspace(0.1, 0.9, len(common_tps))]))

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)
for ax, (hi, title) in zip(axes, [('CK',    'CK (control)'),
                                    ('CKp25', 'CKp25 (observed)'),
                                    ('CF',    'CF (counterfactual)')]):
    ax.scatter(u1[traj_label != hi], u2[traj_label != hi],
               c='#cccccc', s=8, alpha=0.2, linewidths=0, rasterized=True)
    for tp in common_tps:
        mask = (traj_label == hi) & (tp_label == tp)
        ax.scatter(u1[mask], u2[mask], c=[tp_colors[tp]], s=10, alpha=0.9,
                   linewidths=0, rasterized=True, label=tp)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('UMAP 1', fontsize=11)
    ax.axis('off')
axes[0].set_ylabel('UMAP 2', fontsize=11)
axes[-1].legend(title='Time point', fontsize=9, title_fontsize=10,
                loc='best', markerscale=2)

plt.suptitle('Joint UMAP of CK / CKp25 / CF trajectories\n'
             '(color = time point, gray = other trajectories)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(out_dir / 'joint_umap_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
from src.pt import EmpiricalMeasure, barycentric_projection_tangent, wasserstein_logmap
from scipy.spatial import cKDTree
from scipy import stats
from statsmodels.stats.multitest import multipletests

# ── Final time-point PCA arrays ───────────────────────────────────────────────
tp_final        = common_tps[-1]
ckp25_final_pca = ckp25_samples[-1]    # (n_src, n_pcs) observed treated
cf_final_pca    = cf_curve[-1].locs    # (n_cf,  n_pcs) counterfactual
ck_final_pca    = control_samples[-1]  # (n_ck,  n_pcs) control
n_src           = len(ckp25_final_pca)
print(f't={tp_final}  |  CKp25: {n_src}  CF: {len(cf_final_pca)}  CK: {len(ck_final_pca)}')

# ── Gene expression matrix at final time point ────────────────────────────────
mask_ckp25 = (obs['condition'] == 'CKp25') & (obs['time_point'] == tp_final)
mask_ck    = (obs['condition'] == 'CK')    & (obs['time_point'] == tp_final)
idx_ckp25  = np.where(mask_ckp25.values)[0]
idx_ck     = np.where(mask_ck.values)[0]

X         = adata_pp.X if not hasattr(adata_pp.X, 'toarray') else adata_pp.X.toarray()
X_ckp25   = X[idx_ckp25]   # (n_src, n_vars)
X_ck      = X[idx_ck]      # (n_ck,  n_vars)

gene_names = np.array(adata_pp.var_names)
gene2idx   = {g: i for i, g in enumerate(gene_names)}
n_genes    = len(gene_names)

# ── Helper: OT transport then nearest-CK expression lookup ───────────────────
def ot_delta(src_pca, dst_pca, dst_weights=None, k=5):
    """Wasserstein logmap from src → dst, transport src cells,
    look up k nearest actual CK cells at transported positions.
    Returns (n_src, n_vars) delta = X_ck_neighbour - X_src."""
    w_src = np.ones(len(src_pca)) / len(src_pca)
    w_dst = dst_weights / dst_weights.sum() if dst_weights is not None \
            else np.ones(len(dst_pca)) / len(dst_pca)
    src_m = EmpiricalMeasure(locs=src_pca, weights=w_src)
    dst_m = EmpiricalMeasure(locs=dst_pca, weights=w_dst)
    tan   = wasserstein_logmap(src_m, dst_m)
    vels  = barycentric_projection_tangent(tan).vels  if tan.coupling is not None else tan.vels
    transported = src_pca + vels
    _, nn  = cKDTree(ck_final_pca).query(transported, k=k)
    X_nn   = X_ck[nn].mean(axis=1)       # (n_src, n_vars)
    return X_nn - X_ckp25, vels

# ── Comparison 1: CKp25 vs CF (what disease removed relative to CF) ───────────
print('Computing OT: CKp25 → CF ...')
cf_weights       = cf_curve[-1].weights
delta_cf, vels_cf = ot_delta(ckp25_final_pca, cf_final_pca, dst_weights=cf_weights)

# ── Comparison 2: CKp25 vs CK (what disease removed relative to control) ──────
print('Computing OT: CKp25 → CK ...')
delta_ck, vels_ck = ot_delta(ckp25_final_pca, ck_final_pca)

# ── Paired Wilcoxon signed-rank test per gene ─────────────────────────────────
def run_wilcoxon(delta_matrix):
    pvals = np.ones(n_genes)
    for g in range(n_genes):
        d = delta_matrix[:, g]
        if np.any(d != 0):
            try:
                pvals[g] = stats.wilcoxon(d, alternative='two-sided', zero_method='wilcox').pvalue
            except Exception:
                pass
    return pvals, multipletests(pvals, method='fdr_bh')[1]

print('Running Wilcoxon (CKp25 vs CF) ...')
pvals_cf,   padj_cf   = run_wilcoxon(delta_cf)
print('Running Wilcoxon (CKp25 vs CK) ...')
pvals_ck,   padj_ck   = run_wilcoxon(delta_ck)

mean_cf = delta_cf.mean(axis=0)
mean_ck = delta_ck.mean(axis=0)

print(f'\nCKp25 vs CF  — sig genes: '
      f'{(padj_cf < 0.05).sum()} (padj<0.05),  {(padj_cf < 0.01).sum()} (padj<0.01)')
print(f'CKp25 vs CK  — sig genes: '
      f'{(padj_ck < 0.05).sum()} (padj<0.05),  {(padj_ck < 0.01).sum()} (padj<0.01)')


In [ ]:
from matplotlib.patches import Patch

key_genes_de = {
    'Homeostatic': ['P2ry12', 'Tmem119', 'Cx3cr1'],
    'DAM':         ['Apoe',   'Trem2',   'Cd68'],
    'Inflammatory':['Il1b',   'Tnf',     'Ccl2'],
    'Complement':  ['C1qa',   'C1qb',    'C1qc'],
}
cat_colors = {
    'Homeostatic': '#2E86AB',
    'DAM':         '#E07B39',
    'Inflammatory':'#D62828',
    'Complement':  '#6A4C93',
}
key_genes_flat = [g for gs in key_genes_de.values() for g in gs]

# ── Side-by-side volcano: CKp25 vs CF  |  CKp25 vs CK ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, mean_d, padj, title in [
    (axes[0], mean_cf, padj_cf, f'CKp25 vs CF  (t={tp_final})'),
    (axes[1], mean_ck, padj_ck, f'CKp25 vs CK  (t={tp_final})'),
]:
    neg_logp = -np.log10(padj + 1e-300)
    sig_up   = (padj < 0.05) & (mean_d >  0.5)
    sig_down = (padj < 0.05) & (mean_d < -0.5)

    ax.scatter(mean_d[~sig_up & ~sig_down], neg_logp[~sig_up & ~sig_down],
               c='#aaaaaa', s=8, alpha=0.4, linewidths=0)
    ax.scatter(mean_d[sig_up],   neg_logp[sig_up],
               c='#D62828', s=12, alpha=0.7, linewidths=0,
               label=f'Higher in CF/CK ({sig_up.sum()})')
    ax.scatter(mean_d[sig_down], neg_logp[sig_down],
               c='#2E86AB', s=12, alpha=0.7, linewidths=0,
               label=f'Higher in CKp25 ({sig_down.sum()})')

    ax.axhline(-np.log10(0.05), c='k', lw=0.6, ls='--', alpha=0.5)
    ax.axvline( 0.5, c='k', lw=0.6, ls='--', alpha=0.5)
    ax.axvline(-0.5, c='k', lw=0.6, ls='--', alpha=0.5)

    for gene in key_genes_flat:
        if gene in gene2idx:
            gi = gene2idx[gene]
            ax.annotate(gene, xy=(mean_d[gi], neg_logp[gi]),
                        xytext=(4, 3), textcoords='offset points',
                        fontsize=8, fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.2', fc='yellow', alpha=0.6))

    ax.set_xlabel('Mean Δlog\u2082(FPKM+1)  [target − CKp25]', fontsize=11)
    ax.set_ylabel('−log₁₀(FDR)', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.25)

plt.suptitle('OT-based DE: observed CKp25 vs counterfactual and vs control\n'
             'Both via Wasserstein logmap + barycentric projection, Wilcoxon signed-rank + FDR',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(out_dir / 'cf_de_volcano.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Key-gene bar chart ────────────────────────────────────────────────────────
bar_labels = [g for cat, gs in key_genes_de.items() for g in gs]
bar_clrs   = [cat_colors[cat] for cat, gs in key_genes_de.items() for g in gs]
y_pos      = np.arange(len(bar_labels))
cf_vals    = [mean_cf[gene2idx[g]] if g in gene2idx else 0. for g in bar_labels]
ck_vals    = [mean_ck[gene2idx[g]] if g in gene2idx else 0. for g in bar_labels]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, vals, xlabel, title in [
    (axes[0], cf_vals, 'Mean Δlog\u2082(FPKM+1)  [CF − CKp25]',  f'CKp25 vs CF  (t={tp_final})'),
    (axes[1], ck_vals, 'Mean Δlog\u2082(FPKM+1)  [CK − CKp25]',  f'CKp25 vs CK  (t={tp_final})'),
]:
    ax.barh(y_pos, vals[::-1], color=bar_clrs[::-1])
    ax.set_yticks(y_pos); ax.set_yticklabels(bar_labels[::-1], fontsize=9)
    ax.axvline(0, c='k', lw=0.8)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_title(title, fontsize=12)
    ax.legend(handles=[Patch(color=c, label=l) for l, c in cat_colors.items()],
              fontsize=8, loc='lower right')

plt.suptitle('Key microglia marker changes  (OT transport, both comparisons)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(out_dir / 'cf_de_key_genes.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Top 20 gene table ─────────────────────────────────────────────────────────
for label, mean_d, padj in [('CKp25 vs CF', mean_cf, padj_cf),
                              ('CKp25 vs CK', mean_ck, padj_ck)]:
    order = np.argsort(np.abs(mean_d))[::-1]
    print(f'\nTop 20  [{label}]  t={tp_final}')
    print(f'{"Rank":>4}  {"Gene":<18}  {"Mean Δ":>10}  {"padj":>10}')
    print('─' * 48)
    shown = 0
    for gi in order:
        if shown >= 20: break
        print(f'{shown+1:>4}  {gene_names[gi]:<18}  {mean_d[gi]:>+10.4f}  {padj[gi]:>10.2e}')
        shown += 1


In [ ]:
# Per-cell delta histograms for top 5 genes — CF and CK overlaid
top_5    = 5
order_cf = np.argsort(np.abs(mean_cf))[::-1]
order_ck = np.argsort(np.abs(mean_ck))[::-1]

# Use union of top-5 from each comparison so both are represented
top_genes_idx = list(dict.fromkeys(list(order_cf[:top_5]) + list(order_ck[:top_5])))
top_genes_idx = top_genes_idx[:top_5]   # keep exactly 5

fig, axes = plt.subplots(1, top_5, figsize=(15, 3.5))
for ax, gi in zip(axes, top_genes_idx):
    v_cf = delta_cf[:, gi]
    v_ck = delta_ck[:, gi]
    all_vals = np.concatenate([v_cf, v_ck])
    bins = np.linspace(all_vals.min(), all_vals.max(), 30)
    ax.hist(v_cf, bins=bins, color='steelblue',  alpha=0.55, edgecolor='none', label='CF')
    ax.hist(v_ck, bins=bins, color='darkorange', alpha=0.55, edgecolor='none', label='CK')
    ax.axvline(0,           color='k',          lw=1.0)
    ax.axvline(v_cf.mean(), color='steelblue',  lw=1.5, linestyle='--',
               label=f'CF mean={v_cf.mean():.3f}')
    ax.axvline(v_ck.mean(), color='darkorange', lw=1.5, linestyle='--',
               label=f'CK mean={v_ck.mean():.3f}')
    ax.set_title(gene_names[gi], fontsize=11, fontweight='bold')
    ax.set_xlabel('Δlog₂(FPKM+1)  [target − CKp25]', fontsize=8)
    ax.set_ylabel('# cells', fontsize=8)
    ax.legend(fontsize=6)
fig.suptitle(f'Per-cell expression change — top {top_5} genes  (CF vs CK overlaid)',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(out_dir / 'top5_histograms_overlaid.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Per-cell delta histograms for key marker genes — CF and CK overlaid
n_cats = len(key_genes_de)
n_per  = max(len(gs) for gs in key_genes_de.values())

fig, axes = plt.subplots(n_cats, n_per, figsize=(4 * n_per, 3.5 * n_cats),
                          constrained_layout=True)

for row, (cat, genes) in enumerate(key_genes_de.items()):
    for col in range(n_per):
        ax = axes[row, col]
        if col >= len(genes):
            ax.set_visible(False)
            continue
        gene = genes[col]
        if gene not in gene2idx:
            ax.text(0.5, 0.5, f'{gene}\n(not found)', ha='center', va='center',
                    transform=ax.transAxes, fontsize=10)
            continue
        gi   = gene2idx[gene]
        v_cf = delta_cf[:, gi]
        v_ck = delta_ck[:, gi]
        all_vals = np.concatenate([v_cf, v_ck])
        bins = np.linspace(all_vals.min(), all_vals.max(), 30)
        ax.hist(v_cf, bins=bins, color='steelblue',  alpha=0.55, edgecolor='none', label='CF')
        ax.hist(v_ck, bins=bins, color='darkorange', alpha=0.55, edgecolor='none', label='CK')
        ax.axvline(0,           color='k',          lw=1.0)
        ax.axvline(v_cf.mean(), color='steelblue',  lw=1.5, linestyle='--',
                   label=f'CF={v_cf.mean():.2f}')
        ax.axvline(v_ck.mean(), color='darkorange', lw=1.5, linestyle='--',
                   label=f'CK={v_ck.mean():.2f}')
        ax.set_title(gene, fontsize=11, fontweight='bold')
        ax.set_xlabel('Δlog₂(FPKM+1)  [target − CKp25]', fontsize=7)
        ax.set_ylabel(f'{cat}\n# cells' if col == 0 else '# cells', fontsize=8)
        ax.legend(fontsize=6)

fig.suptitle('Per-cell expression change — key marker genes  (CF vs CK overlaid)',
             fontsize=13, fontweight='bold')
plt.savefig(out_dir / 'marker_histograms_overlaid.png', dpi=150, bbox_inches='tight')
plt.show()
